In [ ]:
import json
import os

import agama
import h5py
import matplotlib.pyplot as plt
import numpy as np
import utilities as ut
from matplotlib.lines import Line2D
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.stats import gaussian_kde, norm

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()


# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [ ]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
# sim_lst = ["m12i"]
sim_dir = "/Users/z5114326/Documents/simulations/"

snap = 600
cont_level = 0.75
seed = 3

lz_grid = np.linspace(-12, 12, 500)
et_grid = np.linspace(-4, 0, 500)
XX, YY = np.meshgrid(lz_grid, et_grid)
grid_positions = np.vstack([XX.ravel(), YY.ravel()])
dz = (lz_grid[1] - lz_grid[0]) * (et_grid[1] - et_grid[0])

In [ ]:
# Limits

feh_lims = [
    {"min": -2.0, "max": -1.5},
    {"min": -1.5, "max": -1.0},
    {"min": -1.0, "max": -0.5},
    {"min": -0.5, "max": 0.0},
]

lg_mfo_lims = [
    {"min": 4.0, "max": 4.5},
    {"min": 4.5, "max": 5.0},
    {"min": 5.0, "max": 5.5},
    {"min": 5.5, "max": 6.0},
    {"min": 6.0, "max": 7.0},
]

mas_bins = [
    r"10$^{4.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{4.5}$",
    r"10$^{4.5}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{5.0}$",
    r"10$^{5.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{5.5}$",
    r"10$^{5.5}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{6.0}$",
    r"10$^{6.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$",
]

feh_bins = [
    r"$-2.0 \leq$ [Fe/H] $< -1.5$",
    r"$-1.5 \leq$ [Fe/H] $< -1.0$",
    r"$-1.0 \leq$ [Fe/H] $< -0.5$",
    r"$-0.5 \leq$ [Fe/H] $< 0.0$",
]

# Functions

In [ ]:
def get_level(Z, cont_level):
    Zf = Z.ravel()
    idx = np.argsort(Zf)[::-1]
    Zs = Zf[idx]
    cdf = np.cumsum(Zs)
    cdf /= cdf[-1]
    return Zs[np.searchsorted(cdf, cont_level)]


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def deserialize_store(store):
    out = {}
    for k, v in store.items():
        out[k] = {eval(key): np.array(val) for key, val in v.items()}
    return out


def sample_from_pdf(Z, XX, YY, N, seed=None):
    Z_flat = Z.ravel()
    Z_flat = Z_flat / Z_flat.sum()

    rng = np.random.default_rng(seed)

    indices = rng.choice(len(Z_flat), size=int(np.ceil(N)), p=Z_flat)
    # indices = rng.choice(len(Z_flat), size=int(N), p=Z_flat)

    x = XX.ravel()[indices]
    y = YY.ravel()[indices]

    dx = XX[0, 1] - XX[0, 0]
    dy = YY[1, 0] - YY[0, 0]

    x += (rng.random(len(x)) - 0.5) * dx
    y += (rng.random(len(y)) - 0.5) * dy

    return x, y


# Mass and FeH Binning

In [ ]:
# for sim in sim_lst:
#     # Define Potential
#     pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
#     pot_nbody = agama.Potential(pot_file)
#     af = agama.ActionFinder(pot_nbody, interp=False)

#     # Circular Boundaries
#     r_c = np.arange(-100, 100, 0.5)
#     xyz_c = np.column_stack((r_c, r_c * 0, r_c * 0))
#     vel_rot = np.sqrt(-r_c * pot_nbody.force(xyz_c)[:, 0])
#     vxyz_c = np.column_stack((vel_rot * 0, vel_rot, vel_rot * 0))
#     posvel_c = np.concatenate((xyz_c, vxyz_c), axis=1)
#     ioms_c = af(posvel_c)
#     jphi_c = ioms_c[:, 2]
#     lz_c = jphi_c
#     ep_c = pot_nbody.potential(xyz_c)
#     ek_c = 0.5 * np.sum(np.array(vxyz_c) ** 2, axis=1)
#     et_c = ek_c + ep_c

#     ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
#     ghost_data = h5py.File(ghost_file, "r")

#     cache_mass_feh_pdf = "../data/sim_data/" + sim + "/" + sim + "_mass_feh_pdf_store.json"
#     cache_mass_feh_ngc = "../data/sim_data/" + sim + "/" + sim + "_mass_feh_n_gc_store.json"

#     pdf_mf_store = deserialize_store(load_json(cache_mass_feh_pdf))
#     n_gc_mf_store = deserialize_store(load_json(cache_mass_feh_ngc))

#     for sur in [True, False]:
#         if sur:
#             lab = "sur"
#             alpha = 0.7
#         else:
#             lab = "dis"
#             alpha = 0.2

#         fig, axs = plt.subplots(nrows=5, ncols=4, figsize=(15, 20))
#         plt.subplots_adjust(hspace=0.075, wspace=0.075)

#         # Column labels
#         for j, label in enumerate(feh_bins):
#             axs[0, j].set_title(label)

#         # Row labels
#         for i, label in enumerate(mas_bins):
#             axs[i, -1].text(
#                 +1.05,
#                 0.5,
#                 label,
#                 transform=axs[i, -1].transAxes,
#                 rotation=270,
#                 va="center",
#                 ha="center",
#                 fontsize=SMALL_SIZE,
#             )

#         for i in range(5):
#             for j in range(4):
#                 ax = axs[i, j]
#                 key = (i, j)

#                 if sur:
#                     if key in pdf_mf_store["is"]:
#                         xs, ys = sample_from_pdf(
#                             pdf_mf_store["is"][key], XX, YY, n_gc_mf_store["is"][key], seed
#                         )
#                         ax.scatter(xs, ys, s=5, c="r", alpha=alpha, label="In-situ")
#                         # test_is = "In-situ: " + str(int(n_gc_mf_store["is"][key]))
#                         test_is = "In-situ: " + str(int(np.ceil(n_gc_mf_store["is"][key])))
#                     else:
#                         test_is = "In-situ: " + str(0)
#                     ax.text(0.05, 0.15, test_is, transform=ax.transAxes, va="top", fontsize=10)

#                     if key in pdf_mf_store["es"]:
#                         xs, ys = sample_from_pdf(
#                             pdf_mf_store["es"][key], XX, YY, n_gc_mf_store["es"][key], seed
#                         )
#                         ax.scatter(xs, ys, s=5, c="b", alpha=alpha, label="Ex-situ")
#                         test_es = "Ex-situ: " + str(int(np.ceil(n_gc_mf_store["es"][key])))
#                     else:
#                         test_es = "Ex-situ: " + str(0)
#                     ax.text(0.05, 0.08, test_es, transform=ax.transAxes, va="top", fontsize=10)

#                 else:
#                     if key in pdf_mf_store["id"]:
#                         xs, ys = sample_from_pdf(
#                             pdf_mf_store["id"][key], XX, YY, n_gc_mf_store["id"][key], seed
#                         )
#                         ax.scatter(xs, ys, s=5, c="r", alpha=alpha)
#                         test_id = "In-situ: " + str(int(np.ceil(n_gc_mf_store["id"][key])))
#                     else:
#                         test_id = "In-situ: " + str(0)
#                     ax.text(0.05, 0.15, test_id, transform=ax.transAxes, va="top", fontsize=10)

#                     if key in pdf_mf_store["ed"]:
#                         xs, ys = sample_from_pdf(
#                             pdf_mf_store["ed"][key], XX, YY, n_gc_mf_store["ed"][key], seed
#                         )
#                         ax.scatter(xs, ys, s=5, c="b", alpha=alpha)
#                         test_ed = "Ex-situ: " + str(int(np.ceil(n_gc_mf_store["ed"][key])))
#                     else:
#                         test_ed = "Ex-situ: " + str(0)
#                     ax.text(0.05, 0.08, test_ed, transform=ax.transAxes, va="top", fontsize=10)

#                 ax.plot(lz_c * 10**-3, et_c * 10**-5, "k--", lw=0.5)
#                 # ax.plot(lz_boundary, et_boundary, c="green", lw=1, zorder=0)

#                 ax.set_xlim(-9, 9)
#                 ax.set_ylim(-3.5, 0)

#                 # only show labels on outer axes
#                 if i == 4:
#                     ax.set_xlabel(r"L$_{\rm z}$ (10$^3$ kpc km s$^{-1}$)")
#                 else:
#                     ax.set_xlabel("")
#                     ax.set_xticklabels([])

#                 if j == 0:
#                     ax.set_ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")
#                 else:
#                     ax.set_ylabel("")
#                     ax.set_yticklabels([])

#         # after the plotting loops
#         legend_elements = [
#             Line2D(
#                 [0], [0], color="r", marker="o", markersize=4, alpha=alpha, linestyle="None", label="In-situ"
#             ),
#             Line2D(
#                 [0], [0], color="b", marker="o", markersize=4, alpha=alpha, linestyle="None", label="Ex-situ"
#             ),
#         ]

#         fig.legend(
#             handles=legend_elements,
#             loc="upper center",
#             bbox_to_anchor=(0.5, 0.915),
#             ncol=2,
#             frameon=True,
#         )

#         save_name = "../data/figures/ioms/" + sim + "_mass_feh_ioms_" + lab + ".pdf"
#         fig.savefig(save_name)

In [ ]:
fmin = -2
fmax = 0

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    frac_in = []
    frac_out = []
    for it_id in ghost_data.keys():
        feh = ghost_data[it_id]["source"]["feh"][()]
        msk = (fmin <= feh) & (feh < fmax)

        frac_in.append(len(feh[msk]) / len(feh))
        frac_out.append(len(feh[~msk]) / len(feh))

    print(sim, "-  in:", np.round(np.mean(frac_in), 2), "+/-", np.round(np.std(frac_in), 2))
    print(sim, "- out:", np.round(np.mean(frac_out), 2), "+/-", np.round(np.std(frac_out), 2))
    print()

In [ ]:
(0.84 + 0.85 + 0.82 + 0.82 + 0.86) / 5

In [ ]:
np.sqrt(0.05**2 + 0.05**2 + 0.05**2 + 0.06**2 + 0.03**2) / 5

In [ ]:
sim = "m12i"
ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
ghost_data = h5py.File(ghost_file, "r")

bins = np.arange(-5, 1.5, 0.5)

feh = np.array([])
for it_id in ghost_data.keys():
    feh_i = ghost_data[it_id]["source"]["feh"][()]
    feh = np.concatenate((feh, feh_i))

feh = ghost_data["it001"]["source"]["feh"][()]
plt.hist(feh, bins=bins)

In [ ]:
for sim in sim_lst:
    # Define Potential
    pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
    pot_nbody = agama.Potential(pot_file)
    af = agama.ActionFinder(pot_nbody, interp=False)

    # Circular Boundaries
    r_c = np.arange(-100, 100, 0.5)
    xyz_c = np.column_stack((r_c, r_c * 0, r_c * 0))
    vel_rot = np.sqrt(-r_c * pot_nbody.force(xyz_c)[:, 0])
    vxyz_c = np.column_stack((vel_rot * 0, vel_rot, vel_rot * 0))
    posvel_c = np.concatenate((xyz_c, vxyz_c), axis=1)
    ioms_c = af(posvel_c)
    jphi_c = ioms_c[:, 2]
    lz_c = jphi_c
    ep_c = pot_nbody.potential(xyz_c)
    ek_c = 0.5 * np.sum(np.array(vxyz_c) ** 2, axis=1)
    et_c = ek_c + ep_c

    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    cache_mass_feh_pdf = "../data/sim_data/" + sim + "/" + sim + "_mass_feh_pdf_store.json"
    cache_mass_feh_ngc = "../data/sim_data/" + sim + "/" + sim + "_mass_feh_n_gc_store.json"

    pdf_mf_store = deserialize_store(load_json(cache_mass_feh_pdf))
    n_gc_mf_store = deserialize_store(load_json(cache_mass_feh_ngc))

    for sur in [True, False]:
        if sur:
            lab = "sur"
            alpha = 0.7
        else:
            lab = "dis"
            alpha = 0.2

        fig, axs = plt.subplots(nrows=5, ncols=4, figsize=(7, 8))
        plt.subplots_adjust(hspace=0.075, wspace=0.075)

        # Column labels
        for j, label in enumerate(feh_bins):
            axs[0, j].set_title(label)

        # Row labels
        for i, label in enumerate(mas_bins):
            axs[i, -1].text(
                +1.10,
                0.5,
                label,
                transform=axs[i, -1].transAxes,
                rotation=270,
                va="center",
                ha="center",
                fontsize=SMALL_SIZE,
            )

        for i in range(5):
            for j in range(4):
                ax = axs[i, j]
                key = (i, j)

                if sur:
                    if key in pdf_mf_store["is"]:
                        xs, ys = sample_from_pdf(
                            pdf_mf_store["is"][key], XX, YY, n_gc_mf_store["is"][key], seed
                        )
                        ax.scatter(xs, ys, s=2, c="r", alpha=alpha, label="in situ")
                        # test_is = "In-situ: " + str(int(n_gc_mf_store["is"][key]))
                        test_is = "in situ: " + str(int(np.ceil(n_gc_mf_store["is"][key])))
                    else:
                        test_is = "in situ: " + str(0)
                    ax.text(0.05, 0.18, test_is, transform=ax.transAxes, va="top")

                    if key in pdf_mf_store["es"]:
                        xs, ys = sample_from_pdf(
                            pdf_mf_store["es"][key], XX, YY, n_gc_mf_store["es"][key], seed
                        )
                        ax.scatter(xs, ys, s=2, c="b", alpha=alpha, label="ex situ")
                        test_es = "ex situ: " + str(int(np.ceil(n_gc_mf_store["es"][key])))
                    else:
                        test_es = "ex situ: " + str(0)
                    ax.text(0.05, 0.10, test_es, transform=ax.transAxes, va="top")

                else:
                    if key in pdf_mf_store["id"]:
                        xs, ys = sample_from_pdf(
                            pdf_mf_store["id"][key], XX, YY, n_gc_mf_store["id"][key], seed
                        )
                        ax.scatter(xs, ys, s=2, c="r", alpha=alpha)
                        test_id = "in situ: " + str(int(np.ceil(n_gc_mf_store["id"][key])))
                    else:
                        test_id = "in situ: " + str(0)
                    ax.text(0.05, 0.18, test_id, transform=ax.transAxes, va="top")

                    if key in pdf_mf_store["ed"]:
                        xs, ys = sample_from_pdf(
                            pdf_mf_store["ed"][key], XX, YY, n_gc_mf_store["ed"][key], seed
                        )
                        ax.scatter(xs, ys, s=2, c="b", alpha=alpha)
                        test_ed = "ex situ: " + str(int(np.ceil(n_gc_mf_store["ed"][key])))
                    else:
                        test_ed = "ex situ: " + str(0)
                    ax.text(0.05, 0.10, test_ed, transform=ax.transAxes, va="top")

                ax.plot(lz_c * 10**-3, et_c * 10**-5, "k--", lw=0.5)
                # ax.plot(lz_boundary, et_boundary, c="green", lw=1, zorder=0)

                ax.set_xlim(-9, 9)
                ax.set_ylim(-3.5, 0)

                # only show labels on outer axes
                if i == 4:
                    ax.set_xlabel(r"L$_{\rm z}$ (10$^3$ kpc km s$^{-1}$)")
                else:
                    ax.set_xlabel("")
                    ax.set_xticklabels([])

                if j == 0:
                    ax.set_ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")
                else:
                    ax.set_ylabel("")
                    ax.set_yticklabels([])

        # after the plotting loops
        legend_elements = [
            Line2D(
                [0], [0], color="r", marker="o", markersize=4, alpha=alpha, linestyle="None", label="in situ"
            ),
            Line2D(
                [0], [0], color="b", marker="o", markersize=4, alpha=alpha, linestyle="None", label="ex situ"
            ),
        ]

        fig.legend(
            handles=legend_elements,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.935),
            ncol=2,
            frameon=True,
        )

        save_name = "../data/figures/ioms/" + sim + "_mass_feh_ioms_" + lab + ".pdf"
        fig.savefig(save_name, bbox_inches="tight")

# Mass Only Binning

In [ ]:
for sim in sim_lst:
    pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
    pot_nbody = agama.Potential(pot_file)
    af = agama.ActionFinder(pot_nbody, interp=False)

    # Circular Boundaries
    r_c = np.arange(-100, 100, 0.5)
    xyz_c = np.column_stack((r_c, r_c * 0, r_c * 0))
    vel_rot = np.sqrt(-r_c * pot_nbody.force(xyz_c)[:, 0])
    vxyz_c = np.column_stack((vel_rot * 0, vel_rot, vel_rot * 0))
    posvel_c = np.concatenate((xyz_c, vxyz_c), axis=1)
    ioms_c = af(posvel_c)
    jphi_c = ioms_c[:, 2]
    lz_c = jphi_c
    ep_c = pot_nbody.potential(xyz_c)
    ek_c = 0.5 * np.sum(np.array(vxyz_c) ** 2, axis=1)
    et_c = ek_c + ep_c

    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    cache_mass_pdf = "../data/sim_data/" + sim + "/" + sim + "_mass_pdf_store.json"
    cache_mass_ngc = "../data/sim_data/" + sim + "/" + sim + "_mass_n_gc_store.json"

    pdf_m_store = deserialize_store(load_json(cache_mass_pdf))
    n_gc_m_store = deserialize_store(load_json(cache_mass_ngc))

    fig, axs = plt.subplots(nrows=2, ncols=5, figsize=(15, 5))
    plt.subplots_adjust(hspace=0.075, wspace=0.075)

    for i in range(5):
        s = 2
        if i in [0, 1]:
            alpha = 0.1
        else:
            alpha = 0.3

        axs[0, i].set_title(mas_bins[i])

        if i in pdf_m_store["is"]:
            xs, ys = sample_from_pdf(pdf_m_store["is"][i], XX, YY, n_gc_m_store["is"][i], seed)
            axs[0, i].scatter(xs, ys, s=s, c="r", alpha=alpha, label="In-situ")
            test_is = "In-situ: " + str(int(np.ceil(n_gc_m_store["is"][i])))
        else:
            test_is = "In-situ: " + str(0)
        axs[0, i].text(0.05, 0.245, test_is, transform=axs[0, i].transAxes, va="top", fontsize=10)

        if i in pdf_m_store["es"]:
            xs, ys = sample_from_pdf(pdf_m_store["es"][i], XX, YY, n_gc_m_store["es"][i], seed)
            axs[0, i].scatter(xs, ys, s=seed, c="b", alpha=alpha, label="Ex-situ")
            test_es = "Ex-situ: " + str(int(np.ceil(n_gc_m_store["es"][i])))
        else:
            test_es = "Ex-situ: " + str(0)
        axs[0, i].text(0.05, 0.135, test_es, transform=axs[0, i].transAxes, va="top", fontsize=10)

        if i in pdf_m_store["id"]:
            xd, yd = sample_from_pdf(pdf_m_store["id"][i], XX, YY, n_gc_m_store["id"][i], seed)
            axs[1, i].scatter(xd, yd, s=s, c="r", alpha=alpha)
            test_id = "In-situ: " + str(int(np.ceil(n_gc_m_store["id"][i])))
        else:
            test_id = "In-situ: " + str(0)
        axs[1, i].text(0.05, 0.245, test_id, transform=axs[1, i].transAxes, va="top", fontsize=10)

        if i in pdf_m_store["ed"]:
            xd, yd = sample_from_pdf(pdf_m_store["ed"][i], XX, YY, n_gc_m_store["ed"][i], seed)
            axs[1, i].scatter(xd, yd, s=s, c="b", alpha=alpha)
            test_ed = "Ex-situ: " + str(int(np.ceil(n_gc_m_store["ed"][i])))
        else:
            test_ed = "Ex-situ: " + str(0)
        axs[1, i].text(0.05, 0.135, test_ed, transform=axs[1, i].transAxes, va="top", fontsize=10)

        axs[0, i].plot(lz_c * 10**-3, et_c * 10**-5, "k--", lw=0.5)
        axs[1, i].plot(lz_c * 10**-3, et_c * 10**-5, "k--", lw=0.5)

        axs[0, i].set_xlim(-9, 9)
        axs[1, i].set_xlim(-9, 9)

        axs[0, i].set_ylim(-3.5, 0)
        axs[1, i].set_ylim(-3.5, 0)

        axs[0, i].set_xlabel("")
        axs[0, i].set_xticklabels([])

        axs[1, i].set_xlabel(r"L$_{\rm z}$ (10$^3$ kpc km s$^{-1}$)")

        if i != 0:
            axs[0, i].set_ylabel("")
            axs[1, i].set_ylabel("")

            axs[0, i].set_yticklabels([])
            axs[1, i].set_yticklabels([])
        else:
            axs[0, i].set_ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")
            axs[1, i].set_ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")

    axs[0, -1].text(
        +1.05,
        0.5,
        "Surviving",
        transform=axs[0, -1].transAxes,
        rotation=270,
        va="center",
        ha="center",
        fontsize=SMALL_SIZE,
    )

    axs[1, -1].text(
        +1.05,
        0.5,
        "Disrupted",
        transform=axs[1, -1].transAxes,
        rotation=270,
        va="center",
        ha="center",
        fontsize=SMALL_SIZE,
    )

    # after the plotting loops
    legend_elements = [
        Line2D([0], [0], color="r", marker="o", markersize=3, linestyle="None", label="In-situ"),
        Line2D([0], [0], color="b", marker="o", markersize=3, linestyle="None", label="Ex-situ"),
    ]

    axs[0, 0].legend(handles=legend_elements, loc="lower right", ncols=1, frameon=False, fontsize=SMALL_SIZE)

    save_name = "../data/figures/ioms/" + sim + "_mass_ioms" + ".pdf"
    fig.savefig(save_name)

In [ ]:
for sur in [True, False]:
    if sur:
        lab = "sur"
        alpha = 0.7
    else:
        lab = "dis"
        alpha = 0.2

    fig, axs = plt.subplots(nrows=5, ncols=5, figsize=(15, 15))
    plt.subplots_adjust(hspace=0.075, wspace=0.075)

    for j, sim in enumerate(sim_lst):
        pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
        pot_nbody = agama.Potential(pot_file)
        af = agama.ActionFinder(pot_nbody, interp=False)

        # Circular Boundaries
        r_c = np.arange(-100, 100, 0.5)
        xyz_c = np.column_stack((r_c, r_c * 0, r_c * 0))
        vel_rot = np.sqrt(-r_c * pot_nbody.force(xyz_c)[:, 0])
        vxyz_c = np.column_stack((vel_rot * 0, vel_rot, vel_rot * 0))
        posvel_c = np.concatenate((xyz_c, vxyz_c), axis=1)
        ioms_c = af(posvel_c)
        jphi_c = ioms_c[:, 2]
        lz_c = jphi_c
        ep_c = pot_nbody.potential(xyz_c)
        ek_c = 0.5 * np.sum(np.array(vxyz_c) ** 2, axis=1)
        et_c = ek_c + ep_c

        ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
        ghost_data = h5py.File(ghost_file, "r")

        cache_mass_pdf = "../data/sim_data/" + sim + "/" + sim + "_mass_pdf_store.json"
        cache_mass_ngc = "../data/sim_data/" + sim + "/" + sim + "_mass_n_gc_store.json"

        pdf_m_store = deserialize_store(load_json(cache_mass_pdf))
        n_gc_m_store = deserialize_store(load_json(cache_mass_ngc))

        for i in range(5):
            if j == 0:
                axs[j, i].set_title(mas_bins[i])

            if sur:
                if i in pdf_m_store["is"]:
                    xs, ys = sample_from_pdf(pdf_m_store["is"][i], XX, YY, n_gc_m_store["is"][i], seed)
                    axs[j, i].scatter(xs, ys, s=5, c="r", alpha=0.7, label="In-situ")
                    test_is = "In-situ: " + str(int(np.ceil(n_gc_m_store["is"][i])))
                else:
                    test_is = "In-situ: " + str(0)
                axs[j, i].text(0.05, 0.245, test_is, transform=axs[j, i].transAxes, va="top", fontsize=10)

                if i in pdf_m_store["es"]:
                    xs, ys = sample_from_pdf(pdf_m_store["es"][i], XX, YY, n_gc_m_store["es"][i], seed)
                    axs[j, i].scatter(xs, ys, s=5, c="b", alpha=0.7, label="Ex-situ")
                    test_es = "Ex-situ: " + str(int(np.ceil(n_gc_m_store["es"][i])))
                else:
                    test_es = "Ex-situ: " + str(0)
                axs[j, i].text(0.05, 0.135, test_es, transform=axs[j, i].transAxes, va="top", fontsize=10)

            else:
                if i in pdf_m_store["id"]:
                    xd, yd = sample_from_pdf(pdf_m_store["id"][i], XX, YY, n_gc_m_store["id"][i], seed)
                    axs[j, i].scatter(xd, yd, s=5, c="r", alpha=0.2)
                    test_id = "In-situ: " + str(int(np.ceil(n_gc_m_store["id"][i])))
                else:
                    test_id = "In-situ: " + str(0)
                axs[j, i].text(0.05, 0.245, test_id, transform=axs[j, i].transAxes, va="top", fontsize=10)

                if i in pdf_m_store["ed"]:
                    xd, yd = sample_from_pdf(pdf_m_store["ed"][i], XX, YY, n_gc_m_store["ed"][i], seed)
                    axs[j, i].scatter(xd, yd, s=5, c="b", alpha=0.2)
                    test_ed = "Ex-situ: " + str(int(np.ceil(n_gc_m_store["ed"][i])))
                else:
                    test_ed = "Ex-situ: " + str(0)
                axs[j, i].text(0.05, 0.135, test_ed, transform=axs[j, i].transAxes, va="top", fontsize=10)

            axs[j, i].plot(lz_c * 10**-3, et_c * 10**-5, "k--", lw=0.5)

            axs[j, i].set_xlim(-9, 9)
            axs[j, i].set_ylim(-3.5, 0)

            if j != 5:
                axs[j, i].set_xlabel("")
                axs[j, i].set_xticklabels([])
            else:
                axs[j, i].set_xlabel(r"L$_{\rm z}$ (10$^3$ kpc km s$^{-1}$)")

            if i != 0:
                axs[j, i].set_ylabel("")
                axs[j, i].set_yticklabels([])
            else:
                axs[j, i].set_ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")

                axs[j, i].text(
                    0.85,
                    0.15,
                    sim,
                    transform=axs[j, i].transAxes,
                    va="center",
                    ha="center",
                    fontsize=SMALL_SIZE,
                )

        # after the plotting loops
        legend_elements = [
            Line2D([0], [0], color="r", marker="o", markersize=3, linestyle="None", label="In-situ"),
            Line2D([0], [0], color="b", marker="o", markersize=3, linestyle="None", label="Ex-situ"),
        ]

        fig.legend(
            handles=legend_elements,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.93),
            ncol=2,
            frameon=True,
        )

        save_name = "../data/figures/ioms/" + "mass_ioms_" + lab + ".pdf"
        fig.savefig(save_name)